# Mean Reversion Trading System
## Advanced Z-Score Backtesting Framework with Regime Filtering & Risk Management

---

## Strategy Overview

This notebook implements a **production-grade mean reversion backtesting system** grounded in quantitative finance principles.
It goes beyond a basic z-score strategy by incorporating:

- **Multi-indicator signal confirmation** (Z-score + Bollinger Bands + RSI)
- **Market regime filtering** (trend detection via ADX to avoid trading against strong trends)
- **ATR-based dynamic stop-losses** (position-size-aware risk management)
- **Volatility-scaled position sizing** (risk a fixed % of equity per trade)
- **Walk-forward optimisation** (out-of-sample parameter validation)
- **Monte Carlo simulation** (bootstrapped return paths for confidence intervals)
- **Benchmark comparison** (strategy vs buy-and-hold)

---

## Mathematical Foundation

### Z-Score
$$z_t = \frac{P_t - \mu_t}{\sigma_t}$$
where $\mu_t$ and $\sigma_t$ are the rolling mean and standard deviation over a lookback window.

### Bollinger Bands
$$\text{Upper}_t = \mu_t + k \cdot \sigma_t \quad \text{Lower}_t = \mu_t - k \cdot \sigma_t$$

### Average True Range (ATR)
$$\text{TR}_t = \max(H_t - L_t,\ |H_t - C_{t-1}|,\ |L_t - C_{t-1}|)$$
$$\text{ATR}_t = \frac{1}{n}\sum_{i=0}^{n-1} \text{TR}_{t-i}$$

### Sharpe Ratio (annualised)
$$S = \frac{\sqrt{252} \cdot \mathbb{E}[r_t]}{\sigma(r_t)}$$

### Sortino Ratio
$$\text{Sortino} = \frac{\sqrt{252} \cdot \mathbb{E}[r_t]}{\sigma_{\text{downside}}(r_t)}$$

---

## Notebook Structure

| Section | Contents |
|---------|----------|
| **1. Setup** | Dependencies, imports, colour palette |
| **2. Configuration** | All parameters in one place |
| **3. Data & Indicators** | OHLCV fetch, Z-score, BB, RSI, ATR, ADX |
| **4. Signal Engine** | Multi-factor entry/exit with regime filter |
| **5. Risk Management** | ATR stops, volatility-scaled position sizing |
| **6. Backtester** | Event-driven engine with full trade accounting |
| **7. Performance Analytics** | 15+ metrics, drawdown analysis, trade log |
| **8. Benchmark Comparison** | Strategy vs buy-and-hold |
| **9. Walk-Forward Optimisation** | Rolling in/out-of-sample parameter search |
| **10. Monte Carlo Simulation** | Bootstrapped equity paths + VaR/CVaR |
| **11. Dashboard** | 6-panel interactive chart |

---
## Section 1 — Setup

Install and import all required libraries.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy -q

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from matplotlib.patches import FancyArrowPatch
from scipy import stats
from itertools import product
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# ── Colour palette ────────────────────────────────────────────
C = {
    'price'    : '#00BFFF',  'ma'       : '#FFA500',
    'equity'   : '#DA70D6',  'bench'    : '#888888',
    'buy'      : '#00FF7F',  'sell'     : '#FF4500',
    'exit'     : '#FFD700',  'stop'     : '#FF1493',
    'zscore'   : '#87CEEB',  'bb_upper' : '#FF6347',
    'bb_lower' : '#00FA9A',  'rsi'      : '#DDA0DD',
    'adx'      : '#F0E68C',  'threshold': '#FF6347',
    'zero'     : '#555555',  'mc'       : '#4169E1',
}

print('Libraries loaded. Colour palette defined.')

---
## Section 2 — Configuration

All parameters are defined here. Change these and re-run the notebook to test any ticker or parameter set.

### Parameter Guide

| Parameter | Typical Range | Effect |
|-----------|--------------|--------|
| `WINDOW` | 10–50 | Shorter = more signals, higher noise |
| `Z_ENTRY` | 1.5–2.5 | Lower = more trades, lower quality |
| `RSI_OVERSOLD` | 20–40 | Filters entries to confirmed reversals |
| `ADX_TREND_THRESH` | 20–30 | Blocks trades during strong trends |
| `ATR_STOP_MULT` | 1.5–3.0 | Wider stops = fewer stop-outs |
| `RISK_PER_TRADE` | 0.005–0.02 | % of equity risked per trade |

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CONFIGURATION — edit this cell only
# ══════════════════════════════════════════════════════════════

# Asset
SYMBOL     = 'AAPL'        # Any yfinance ticker
START_DATE = '2019-01-01'
END_DATE   = '2024-01-01'

# Core strategy
WINDOW        = 20     # Rolling lookback window (days)
Z_ENTRY       = 2.0    # Z-score threshold for entry
BB_STD        = 2.0    # Bollinger Band width (std deviations)

# Signal confirmation filters
RSI_PERIOD      = 14   # RSI lookback
RSI_OVERSOLD    = 35   # RSI must be below this to confirm long entry
RSI_OVERBOUGHT  = 65   # RSI must be above this to confirm short entry
ADX_PERIOD      = 14   # ADX lookback
ADX_TREND_THRESH= 25   # Skip trades when ADX > this (strong trend)

# Risk management
ATR_PERIOD      = 14   # ATR lookback for stop calculation
ATR_STOP_MULT   = 2.0  # Stop = entry +/- ATR_STOP_MULT * ATR
RISK_PER_TRADE  = 0.01 # Risk 1% of equity per trade
MAX_POSITION    = 0.20 # Never allocate more than 20% to one trade

# Transaction costs
FEE             = 0.001  # 0.1% per trade (commission + slippage)

# Walk-forward optimisation
WFO_TRAIN_MONTHS = 18   # Training window
WFO_TEST_MONTHS  =  6   # Out-of-sample window

# Monte Carlo
MC_SIMULATIONS  = 500   # Number of bootstrapped equity paths
MC_BLOCK_SIZE   = 10    # Block size for block bootstrap (days)

# ══════════════════════════════════════════════════════════════

print(f'Configuration set for {SYMBOL}  |  {START_DATE} to {END_DATE}')

---
## Section 3 — Data & Technical Indicators

We compute **six indicators** from raw OHLCV data:

| Indicator | Purpose |
|-----------|--------|
| **Rolling Z-score** | Primary entry/exit signal |
| **Bollinger Bands** | Confirm price is at statistical extreme |
| **RSI** | Momentum confirmation — avoid entries into momentum |
| **ATR** | Measure volatility for stop placement |
| **ADX** | Trend strength — skip trades in strong trending markets |
| **Realised Volatility** | Annualised 20-day rolling vol for position sizing |

In [ ]:
def fetch_ohlcv(symbol, start, end):
    """Download full OHLCV and normalise columns."""
    raw = yf.download(symbol, start=start, end=end, progress=False, auto_adjust=True)
    if raw.empty:
        raise ValueError(f'No data for {symbol}. Check ticker/dates.')
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = [col[0] for col in raw.columns]
    return raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()


def compute_rsi(close, period=14):
    delta  = close.diff()
    gain   = delta.clip(lower=0).rolling(period).mean()
    loss   = (-delta.clip(upper=0)).rolling(period).mean()
    rs     = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def compute_atr(high, low, close, period=14):
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low  - close.shift(1)).abs(),
    ], axis=1).max(axis=1)
    return tr.rolling(period).mean()


def compute_adx(high, low, close, period=14):
    """Average Directional Index — measures trend strength, not direction."""
    up   = high.diff()
    down = -low.diff()
    plus_dm  = np.where((up > down) & (up > 0), up, 0.0)
    minus_dm = np.where((down > up) & (down > 0), down, 0.0)

    atr      = compute_atr(high, low, close, period)
    plus_di  = 100 * pd.Series(plus_dm,  index=close.index).rolling(period).mean() / atr
    minus_di = 100 * pd.Series(minus_dm, index=close.index).rolling(period).mean() / atr

    dx  = (100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan))
    adx = dx.rolling(period).mean()
    return adx, plus_di, minus_di


def add_indicators(ohlcv, window, bb_std, rsi_period, atr_period, adx_period):
    """Compute all indicators and return enriched DataFrame."""
    df = ohlcv.copy()
    c  = df['Close']

    # Rolling mean / std / z-score
    df['rolling_mean'] = c.rolling(window).mean()
    df['rolling_std']  = c.rolling(window).std()
    df['zscore']       = (c - df['rolling_mean']) / df['rolling_std']

    # Bollinger Bands
    df['bb_upper'] = df['rolling_mean'] + bb_std * df['rolling_std']
    df['bb_lower'] = df['rolling_mean'] - bb_std * df['rolling_std']
    df['bb_pct']   = (c - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])  # 0–1

    # RSI
    df['rsi'] = compute_rsi(c, rsi_period)

    # ATR
    df['atr'] = compute_atr(df['High'], df['Low'], c, atr_period)

    # ADX
    df['adx'], df['plus_di'], df['minus_di'] = compute_adx(
        df['High'], df['Low'], c, adx_period
    )

    # Realised volatility (annualised)
    log_ret        = np.log(c / c.shift(1))
    df['realised_vol'] = log_ret.rolling(window).std() * np.sqrt(252)

    # Log returns for performance calc
    df['log_return'] = log_ret

    return df.dropna().copy()


# ── Fetch + compute ──────────────────────────────────────────
ohlcv = fetch_ohlcv(SYMBOL, START_DATE, END_DATE)
df    = add_indicators(ohlcv, WINDOW, BB_STD, RSI_PERIOD, ATR_PERIOD, ADX_PERIOD)

print(f'Rows after indicator warmup : {len(df)}')
print(f'Date range                  : {df.index[0].date()} — {df.index[-1].date()}')
print(f'Average realised vol        : {df["realised_vol"].mean():.1%}')
print(f'Average ATR                 : ${df["atr"].mean():.2f}')
print(f'Average ADX                 : {df["adx"].mean():.1f}')
df[['Close','zscore','rsi','atr','adx','realised_vol']].tail(8).round(3)

---
## Section 4 — Multi-Factor Signal Engine

### Entry Logic

A **long entry** requires ALL of the following to be true simultaneously:
1. Z-score crosses below −`Z_ENTRY` (price statistically cheap)
2. Price is below the lower Bollinger Band (confirms extremity)
3. RSI < `RSI_OVERSOLD` (momentum is oversold — reversal likely)
4. ADX < `ADX_TREND_THRESH` (no strong prevailing trend — mean reversion valid)

A **short entry** requires the mirror conditions.

### Exit Logic

A position is closed when **any** of the following occurs:
- Z-score reverts back through ±0.5 (mean reversion achieved)
- ATR-based stop-loss is hit
- RSI reaches neutral zone (momentum exhausted)

In [ ]:
def generate_signals(df, z_entry, rsi_oversold, rsi_overbought, adx_thresh):
    """
    Multi-factor signal generation.
    Returns DataFrame with 'signal' (1/-1/0) and 'exit' (1/0) columns.
    """
    df = df.copy()
    prev_z   = df['zscore'].shift(1)
    prev_rsi = df['rsi'].shift(1)

    # ── Long entry conditions ─────────────────────────────────
    long_zscore = (prev_z >= -z_entry) & (df['zscore'] < -z_entry)  # crossover
    long_bb     = df['Close'] < df['bb_lower']                        # below BB
    long_rsi    = df['rsi'] < rsi_oversold                            # oversold
    long_regime = df['adx'] < adx_thresh                              # no trend

    # ── Short entry conditions ────────────────────────────────
    short_zscore = (prev_z <= z_entry) & (df['zscore'] > z_entry)
    short_bb     = df['Close'] > df['bb_upper']
    short_rsi    = df['rsi'] > rsi_overbought
    short_regime = df['adx'] < adx_thresh

    df['signal'] = 0
    df.loc[long_zscore  & long_bb  & long_rsi  & long_regime,  'signal'] =  1
    df.loc[short_zscore & short_bb & short_rsi & short_regime, 'signal'] = -1

    # ── Exit conditions ───────────────────────────────────────
    # Reversion: z-score crosses back through ±0.5
    long_revert  = (df['zscore'].shift(1) <= 0.5) & (df['zscore'] > 0.5)
    short_revert = (df['zscore'].shift(1) >= -0.5) & (df['zscore'] < -0.5)
    # RSI neutralising
    rsi_neutral  = (df['rsi'] > 45) & (df['rsi'] < 55)

    df['exit'] = 0
    df.loc[long_revert | short_revert | rsi_neutral, 'exit'] = 1

    # Signal counts
    n_long  = (df['signal'] ==  1).sum()
    n_short = (df['signal'] == -1).sum()
    print(f'Long entries  : {n_long}')
    print(f'Short entries : {n_short}')
    print(f'Total signals : {n_long + n_short}')
    if n_long + n_short == 0:
        print('  ⚠ No signals — try loosening RSI or ADX thresholds.')

    return df


df = generate_signals(df, Z_ENTRY, RSI_OVERSOLD, RSI_OVERBOUGHT, ADX_TREND_THRESH)

---
## Section 5 — Risk Management

### Volatility-Scaled Position Sizing

Instead of investing a fixed dollar amount, we risk a **fixed percentage of equity** per trade.
The position size is determined by how far the stop-loss is from the entry:

$$\text{Position Size} = \frac{\text{Risk Per Trade} \times \text{Equity}}{\text{Stop Distance}}$$

$$\text{Stop Distance} = \text{ATR\_STOP\_MULT} \times \text{ATR}_t$$

This means:
- In **low volatility** regimes: larger position (stop is tight)
- In **high volatility** regimes: smaller position (stop is wide)

The position is also capped at `MAX_POSITION` to prevent concentration risk.

In [ ]:
def compute_position_size(equity, atr, risk_per_trade, atr_stop_mult, max_position, price):
    """
    Kelly-inspired volatility-scaled position sizing.

    Returns fraction of equity to allocate (0 to max_position).
    """
    stop_distance = atr_stop_mult * atr          # dollar distance to stop
    if stop_distance <= 0 or price <= 0:
        return 0.0
    dollar_risk   = equity * risk_per_trade       # $ we are willing to lose
    shares        = dollar_risk / stop_distance   # shares to hold
    alloc_frac    = (shares * price) / equity     # as fraction of equity
    return float(np.clip(alloc_frac, 0, max_position))


def compute_stop_price(entry_price, direction, atr, atr_stop_mult):
    """
    Return the ATR-based stop price for an open position.
    Long:  stop = entry - mult * ATR
    Short: stop = entry + mult * ATR
    """
    stop_dist = atr_stop_mult * atr
    return entry_price - direction * stop_dist


print('Position sizing and stop functions defined.')

# Preview: what would position size be at current ATR and equity?
sample_atr   = df['atr'].iloc[-1]
sample_price = df['Close'].iloc[-1]
sample_size  = compute_position_size(10000, sample_atr, RISK_PER_TRADE,
                                      ATR_STOP_MULT, MAX_POSITION, sample_price)
print(f'\nSample (equity=$10,000, ATR=${sample_atr:.2f}):')
print(f'  Position size  : {sample_size:.1%} of equity')
print(f'  Dollar amount  : ${10000*sample_size:,.0f}')
print(f'  Stop distance  : ${ATR_STOP_MULT * sample_atr:.2f}')

---
## Section 6 — Event-Driven Backtester

The backtester processes each bar sequentially and maintains full position state:

- **Entry**: checks signal, computes position size, sets ATR stop
- **Stop check**: on every bar, checks if price has breached the stop
- **Exit**: on reversal signal, closes at close price
- **Accounting**: tracks equity, applies fees on both sides

> All trades execute at the **close price** of the signal bar — a conservative assumption that avoids lookahead bias.

In [ ]:
def run_backtest(df, fee, risk_per_trade, atr_stop_mult, max_position):
    """
    Full event-driven backtester with ATR stops and position sizing.

    Returns: dict with equity curve, trade log, drawdown series.
    """
    df           = df.copy()
    equity       = 1.0
    position     = 0          # 1 = long, -1 = short, 0 = flat
    entry_price  = 0.0
    stop_price   = 0.0
    pos_size     = 0.0        # fraction of equity in position
    equity_curve = []
    trades       = []

    for i in range(len(df)):
        row    = df.iloc[i]
        price  = row['Close']
        signal = row['signal']
        exit_  = row['exit']
        atr    = row['atr']

        # ── Stop loss check (before entry logic) ──────────────
        if position != 0:
            stop_hit = (
                (position ==  1 and price <= stop_price) or
                (position == -1 and price >= stop_price)
            )
            if stop_hit:
                pnl      = position * (stop_price - entry_price) / entry_price
                equity  *= (1 + pnl * pos_size) * (1 - fee)
                trades[-1].update({
                    'exit_price': stop_price,
                    'exit_type' : 'stop',
                    'pnl_pct'   : round(pnl * 100, 3),
                    'exit_bar'  : i,
                })
                position = 0
                equity_curve.append(equity)
                continue

        # ── Entry ─────────────────────────────────────────────
        if position == 0 and signal != 0:
            pos_size    = compute_position_size(
                equity, atr, risk_per_trade, atr_stop_mult, max_position, price
            )
            if pos_size > 0:
                position    = signal
                entry_price = price
                stop_price  = compute_stop_price(price, signal, atr, atr_stop_mult)
                equity     *= (1 - fee)    # entry cost
                trades.append({
                    'direction' : 'long' if signal == 1 else 'short',
                    'entry_price': price,
                    'stop_price' : stop_price,
                    'pos_size'   : round(pos_size, 4),
                    'entry_bar'  : i,
                    'entry_date' : df.index[i],
                    'entry_atr'  : round(atr, 3),
                })

        # ── Signal exit ───────────────────────────────────────
        elif position != 0 and exit_:
            pnl      = position * (price - entry_price) / entry_price
            equity  *= (1 + pnl * pos_size) * (1 - fee)
            trades[-1].update({
                'exit_price': price,
                'exit_type' : 'signal',
                'pnl_pct'   : round(pnl * 100, 3),
                'exit_bar'  : i,
            })
            position = 0

        equity_curve.append(equity)

    # ── Assemble results ──────────────────────────────────────
    df = df.copy()
    df['equity']   = equity_curve
    df['drawdown'] = (df['equity'] / df['equity'].cummax()) - 1

    closed   = [t for t in trades if 'pnl_pct' in t]
    stops    = [t for t in closed if t.get('exit_type') == 'stop']
    win_rate = (
        sum(1 for t in closed if t['pnl_pct'] > 0) / len(closed) * 100
        if closed else 0.0
    )

    return {
        'df'        : df,
        'trades'    : trades,
        'closed'    : closed,
        'n_trades'  : len(closed),
        'n_stops'   : len(stops),
        'win_rate'  : win_rate,
    }


results = run_backtest(df, FEE, RISK_PER_TRADE, ATR_STOP_MULT, MAX_POSITION)

print(f'Backtest complete')
print(f'Closed trades  : {results["n_trades"]}')
print(f'Stop-outs      : {results["n_stops"]}')
print(f'Win rate       : {results["win_rate"]:.1f}%')

---
## Section 7 — Performance Analytics

We compute **15+ professional performance metrics** used by quantitative funds:

| Metric | Formula | What it measures |
|--------|---------|------------------|
| **CAGR** | $(V_T/V_0)^{252/T} - 1$ | Compound annual growth rate |
| **Sharpe** | $\sqrt{252} \cdot \mu_r / \sigma_r$ | Risk-adjusted return |
| **Sortino** | $\sqrt{252} \cdot \mu_r / \sigma_{\text{down}}$ | Downside-risk-adjusted return |
| **Calmar** | $\text{CAGR} / |\text{MaxDD}|$ | Return per unit of max drawdown |
| **Max Drawdown** | $\min((V_t - \max V_s) / \max V_s)$ | Worst peak-to-trough loss |
| **Win Rate** | Winning trades / total trades | Percentage of profitable trades |
| **Profit Factor** | Gross profit / gross loss | How much profit per unit of loss |
| **Expectancy** | $W \cdot \text{AvgWin} + L \cdot \text{AvgLoss}$ | Expected PnL per trade |

In [ ]:
def compute_metrics(results, symbol=''):
    """Compute comprehensive performance metrics."""
    df     = results['df']
    closed = results['closed']

    equity  = df['equity']
    returns = equity.pct_change().dropna()
    n_days  = len(equity)

    # Return metrics
    total_return = equity.iloc[-1] - 1
    cagr         = (equity.iloc[-1] ** (252 / n_days)) - 1

    # Risk metrics
    ann_vol      = returns.std() * np.sqrt(252)
    sharpe       = (cagr / ann_vol) if ann_vol > 0 else 0
    downside     = returns[returns < 0].std() * np.sqrt(252)
    sortino      = (cagr / downside) if downside > 0 else 0

    roll_max     = equity.cummax()
    drawdown_ser = (equity - roll_max) / roll_max
    max_dd       = drawdown_ser.min()
    calmar       = (cagr / abs(max_dd)) if max_dd != 0 else 0

    # Average drawdown duration
    in_dd       = drawdown_ser < 0
    dd_duration = in_dd.astype(int).groupby((~in_dd).cumsum()).sum()
    avg_dd_dur  = dd_duration[dd_duration > 0].mean() if (dd_duration > 0).any() else 0

    # Trade-level metrics
    if closed:
        pnls      = [t['pnl_pct'] for t in closed]
        wins      = [p for p in pnls if p > 0]
        losses    = [p for p in pnls if p < 0]
        win_rate  = len(wins) / len(pnls) * 100
        avg_win   = np.mean(wins)   if wins   else 0
        avg_loss  = np.mean(losses) if losses else 0
        pf        = (sum(wins) / abs(sum(losses))) if losses else np.inf
        expectancy = (win_rate/100) * avg_win + (1 - win_rate/100) * avg_loss
    else:
        win_rate = avg_win = avg_loss = pf = expectancy = 0

    # VaR / CVaR
    var_95  = np.percentile(returns, 5)
    cvar_95 = returns[returns <= var_95].mean()

    metrics = {
        'Total Return'     : total_return,
        'CAGR'             : cagr,
        'Ann. Volatility'  : ann_vol,
        'Sharpe Ratio'     : sharpe,
        'Sortino Ratio'    : sortino,
        'Calmar Ratio'     : calmar,
        'Max Drawdown'     : max_dd,
        'Avg DD Duration'  : avg_dd_dur,
        'VaR (95%)'        : var_95,
        'CVaR (95%)'       : cvar_95,
        'N Trades'         : results['n_trades'],
        'N Stop-outs'      : results['n_stops'],
        'Win Rate'         : win_rate / 100,
        'Avg Win'          : avg_win / 100,
        'Avg Loss'         : avg_loss / 100,
        'Profit Factor'    : pf,
        'Expectancy'       : expectancy / 100,
    }

    # ── Print ─────────────────────────────────────────────────
    print(f'{'═'*48}')
    print(f'  PERFORMANCE SUMMARY  —  {symbol}')
    print(f'{'═'*48}')
    print(f'  Total Return     : {total_return*100:>10.2f}%')
    print(f'  CAGR             : {cagr*100:>10.2f}%')
    print(f'  Ann. Volatility  : {ann_vol*100:>10.2f}%')
    print(f'{'─'*48}')
    print(f'  Sharpe Ratio     : {sharpe:>10.3f}')
    print(f'  Sortino Ratio    : {sortino:>10.3f}')
    print(f'  Calmar Ratio     : {calmar:>10.3f}')
    print(f'{'─'*48}')
    print(f'  Max Drawdown     : {max_dd*100:>10.2f}%')
    print(f'  Avg DD Duration  : {avg_dd_dur:>10.1f} days')
    print(f'  VaR (95%, 1d)    : {var_95*100:>10.3f}%')
    print(f'  CVaR (95%, 1d)   : {cvar_95*100:>10.3f}%')
    print(f'{'─'*48}')
    print(f'  Trades           : {results["n_trades"]:>10}')
    print(f'  Stop-outs        : {results["n_stops"]:>10}')
    print(f'  Win Rate         : {win_rate:>10.1f}%')
    print(f'  Avg Win          : {avg_win:>10.3f}%')
    print(f'  Avg Loss         : {avg_loss:>10.3f}%')
    print(f'  Profit Factor    : {pf:>10.3f}')
    print(f'  Expectancy/trade : {expectancy:>10.3f}%')
    print(f'{'═'*48}')

    return metrics


metrics = compute_metrics(results, SYMBOL)

### Trade Log

Full record of every closed trade.

In [ ]:
if results['closed']:
    tlog = pd.DataFrame(results['closed'])
    tlog = tlog[['direction','entry_date','entry_price','exit_price',
                 'stop_price','exit_type','pos_size','pnl_pct']].copy()
    tlog.columns = ['Dir','Entry Date','Entry $','Exit $','Stop $','Exit Type','Size','PnL %']
    tlog['PnL %'] = tlog['PnL %'].apply(lambda x: f'+{x:.3f}%' if x > 0 else f'{x:.3f}%')
    tlog['Size']  = tlog['Size'].apply(lambda x: f'{x:.1%}')
    tlog.index    = range(1, len(tlog)+1)
    tlog
else:
    print('No closed trades. Try loosening RSI or ADX thresholds in Section 2.')

---
## Section 8 — Benchmark Comparison

We compare the strategy against a **buy-and-hold** benchmark on the same asset.
This is the minimum bar any active strategy must clear — if it cannot beat passive holding, there is no justification for the added complexity and transaction costs.

In [ ]:
def compute_benchmark(df):
    """Buy-and-hold equity curve, normalised to start at 1."""
    bm = df['Close'] / df['Close'].iloc[0]
    return bm


res_df    = results['df']
benchmark = compute_benchmark(res_df)

strat_ret = res_df['equity'].iloc[-1] - 1
bench_ret = benchmark.iloc[-1] - 1

strat_vol = res_df['equity'].pct_change().dropna().std() * np.sqrt(252)
bench_vol = benchmark.pct_change().dropna().std() * np.sqrt(252)

print(f'{'─'*42}')
print(f'{'':20s}  {'Strategy':>9}  {'Buy&Hold':>9}')
print(f'{'─'*42}')
print(f'  Total Return   :  {strat_ret*100:>8.2f}%  {bench_ret*100:>8.2f}%')
print(f'  Ann. Vol       :  {strat_vol*100:>8.2f}%  {bench_vol*100:>8.2f}%')
print(f'  Max Drawdown   :  {res_df["drawdown"].min()*100:>8.2f}%')
print(f'{'─'*42}')

alpha = strat_ret - bench_ret
print(f'  Alpha vs benchmark: {alpha*100:+.2f}%')

---
## Section 9 — Walk-Forward Optimisation

In-sample optimisation suffers from **overfitting** — parameters that worked historically may not generalise.

Walk-forward optimisation addresses this by:
1. Splitting the data into rolling **train** and **test** windows
2. Finding the best parameters (window, z_entry) on the train window
3. Applying those parameters to the **out-of-sample** test window
4. Rolling the windows forward and repeating

The out-of-sample Sharpe ratio is the true measure of strategy robustness.

> ⏱ This cell is computationally intensive. Reduce the parameter grid to speed it up.

In [ ]:
# Parameter grid to search
PARAM_GRID = {
    'window'  : [10, 15, 20, 30],
    'z_entry' : [1.5, 2.0, 2.5],
}

def wfo_sharpe(df_slice, window, z_entry):
    """Run a fast backtest on a slice and return Sharpe ratio."""
    try:
        d = add_indicators(df_slice, window, BB_STD, RSI_PERIOD, ATR_PERIOD, ADX_PERIOD)
        d = generate_signals(d, z_entry, RSI_OVERSOLD, RSI_OVERBOUGHT, ADX_TREND_THRESH)
        r = run_backtest(d, FEE, RISK_PER_TRADE, ATR_STOP_MULT, MAX_POSITION)
        eq  = r['df']['equity']
        ret = eq.pct_change().dropna()
        return (np.sqrt(252) * ret.mean() / ret.std()) if ret.std() > 0 else -99
    except Exception:
        return -99


# Walk-forward loop
train_days = WFO_TRAIN_MONTHS * 21
test_days  = WFO_TEST_MONTHS  * 21
step       = test_days
n_rows     = len(ohlcv)

wfo_results = []
start_idx   = 0

while start_idx + train_days + test_days <= n_rows:
    train_slice = ohlcv.iloc[start_idx : start_idx + train_days]
    test_slice  = ohlcv.iloc[start_idx + train_days : start_idx + train_days + test_days]

    # Find best params on train
    best_sharpe, best_params = -99, {}
    for w, z in product(PARAM_GRID['window'], PARAM_GRID['z_entry']):
        s = wfo_sharpe(train_slice, w, z)
        if s > best_sharpe:
            best_sharpe, best_params = s, {'window': w, 'z_entry': z}

    # Apply best params out-of-sample
    oos_sharpe = wfo_sharpe(test_slice, **best_params)

    wfo_results.append({
        'train_start' : ohlcv.index[start_idx].date(),
        'test_start'  : ohlcv.index[start_idx + train_days].date(),
        'best_window' : best_params.get('window'),
        'best_z'      : best_params.get('z_entry'),
        'train_sharpe': round(best_sharpe, 3),
        'oos_sharpe'  : round(oos_sharpe,  3),
    })
    start_idx += step

wfo_df = pd.DataFrame(wfo_results)
print(f'Walk-forward windows: {len(wfo_df)}')
print(f'Avg OOS Sharpe      : {wfo_df["oos_sharpe"].mean():.3f}')
print(f'OOS Sharpe > 0      : {(wfo_df["oos_sharpe"] > 0).mean():.0%} of windows')
wfo_df

---
## Section 10 — Monte Carlo Simulation

Monte Carlo analysis answers: **how robust is this equity curve?**

We use **block bootstrapping** — resampling blocks of consecutive daily returns (preserving autocorrelation) to generate 500 alternative equity paths.

From these paths we compute:
- **Median** outcome
- **P5 / P95** confidence bands
- **Value at Risk (VaR)** — 5th percentile of 1-year returns
- **Conditional VaR (CVaR)** — expected return in the worst 5% of scenarios
- **Probability of ruin** — paths that lose more than 50% of equity

In [ ]:
def block_bootstrap(returns, n_sims, block_size, n_periods):
    """
    Block bootstrap: resample consecutive blocks of returns to
    preserve short-term autocorrelation structure.
    Returns array of shape (n_periods, n_sims).
    """
    returns  = np.array(returns)
    n        = len(returns)
    n_blocks = int(np.ceil(n_periods / block_size))
    paths    = np.zeros((n_periods, n_sims))

    for sim in range(n_sims):
        starts  = np.random.randint(0, n - block_size, size=n_blocks)
        sampled = np.concatenate([returns[s:s+block_size] for s in starts])
        paths[:, sim] = sampled[:n_periods]

    return paths


# Strategy daily returns
strat_returns = results['df']['equity'].pct_change().dropna().values
n_periods     = len(strat_returns)

# Bootstrap
print(f'Running {MC_SIMULATIONS} Monte Carlo paths ({n_periods} days each)...')
mc_returns = block_bootstrap(strat_returns, MC_SIMULATIONS, MC_BLOCK_SIZE, n_periods)

# Build equity curves from bootstrapped returns
mc_equity = np.cumprod(1 + mc_returns, axis=0)  # shape: (n_periods, n_sims)

# Terminal values
terminal       = mc_equity[-1, :]
mc_total_ret   = terminal - 1

p5, p50, p95   = np.percentile(terminal, [5, 50, 95])
var_mc         = np.percentile(mc_total_ret, 5)
cvar_mc        = mc_total_ret[mc_total_ret <= var_mc].mean()
prob_ruin      = (terminal < 0.5).mean()    # lose >50%
prob_profit    = (terminal > 1.0).mean()

print(f'\n{'─'*42}')
print(f'  MONTE CARLO RESULTS ({MC_SIMULATIONS} paths)')
print(f'{'─'*42}')
print(f'  Median terminal equity : {p50:.3f}x')
print(f'  P5  terminal equity    : {p5:.3f}x')
print(f'  P95 terminal equity    : {p95:.3f}x')
print(f'  VaR  (5%, 1-yr)        : {var_mc*100:.2f}%')
print(f'  CVaR (5%, 1-yr)        : {cvar_mc*100:.2f}%')
print(f'  Prob of >50% loss      : {prob_ruin:.1%}')
print(f'  Prob of any profit     : {prob_profit:.1%}')
print(f'{'─'*42}')

---
## Section 11 — Dashboard

6-panel chart:
1. **Price + Bollinger Bands** with long/short/exit markers
2. **Z-score** with shaded entry zones
3. **RSI + ADX** — signal confirmation indicators
4. **Equity Curve** vs buy-and-hold benchmark
5. **Drawdown** — time spent underwater
6. **Monte Carlo** — bootstrapped equity paths with confidence bands

In [ ]:
plt.style.use('dark_background')

res_df    = results['df']
trades    = results['trades']
closed    = results['closed']
benchmark = compute_benchmark(res_df)

long_idx  = [t['entry_bar'] for t in trades if t['direction'] == 'long'  and 'pnl_pct' in t]
shrt_idx  = [t['entry_bar'] for t in trades if t['direction'] == 'short' and 'pnl_pct' in t]
exit_idx  = [t['exit_bar']  for t in closed]
stop_idx  = [t['exit_bar']  for t in closed if t.get('exit_type') == 'stop']

fig = plt.figure(figsize=(18, 22))
fig.patch.set_facecolor('#0a0a0a')
gs  = gridspec.GridSpec(6, 1, figure=fig, hspace=0.55,
                        height_ratios=[2.5, 1.2, 1.2, 2.0, 1.0, 2.0])

axes = [fig.add_subplot(gs[i]) for i in range(6)]
if len(axes) > 1:
    for ax in axes[1:2]:
        ax.sharex(axes[0])

for ax in axes:
    ax.set_facecolor('#0f0f0f')
    ax.tick_params(colors='#888888', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#222222')

# ── Panel 1: Price + Bollinger Bands ─────────────────────────
ax = axes[0]
ax.plot(res_df.index, res_df['Close'],        color=C['price'],    lw=1.2, label='Price', zorder=3)
ax.plot(res_df.index, res_df['rolling_mean'], color=C['ma'],       lw=0.9, linestyle='--', alpha=0.8, label=f'{WINDOW}d Mean')
ax.plot(res_df.index, res_df['bb_upper'],     color=C['bb_upper'], lw=0.7, linestyle=':',  alpha=0.7, label='BB Upper')
ax.plot(res_df.index, res_df['bb_lower'],     color=C['bb_lower'], lw=0.7, linestyle=':',  alpha=0.7, label='BB Lower')
ax.fill_between(res_df.index, res_df['bb_upper'], res_df['bb_lower'], alpha=0.04, color='white')

if long_idx:
    ax.scatter(res_df.iloc[long_idx].index, res_df.iloc[long_idx]['Close'],
               marker='^', color=C['buy'],  s=100, zorder=5, label='Long entry')
if shrt_idx:
    ax.scatter(res_df.iloc[shrt_idx].index, res_df.iloc[shrt_idx]['Close'],
               marker='v', color=C['sell'], s=100, zorder=5, label='Short entry')
if exit_idx:
    sig_exit = [i for i in exit_idx if i not in stop_idx]
    if sig_exit:
        ax.scatter(res_df.iloc[sig_exit].index, res_df.iloc[sig_exit]['Close'],
                   marker='o', color=C['exit'], s=60, zorder=5, label='Signal exit')
if stop_idx:
    ax.scatter(res_df.iloc[stop_idx].index, res_df.iloc[stop_idx]['Close'],
               marker='x', color=C['stop'], s=80, zorder=5, linewidths=2, label='Stop-out')

ax.set_ylabel('Price (USD)', color='#aaaaaa', fontsize=9)
ax.set_title(f'Mean Reversion System  —  {SYMBOL}  |  {START_DATE} to {END_DATE}',
             color='white', fontsize=13, pad=12)
ax.legend(fontsize=7, loc='upper left', facecolor='#1a1a1a', edgecolor='#333', labelcolor='white', ncol=4)
ax.grid(alpha=0.08)

# ── Panel 2: Z-score ──────────────────────────────────────────
ax = axes[1]
ax.plot(res_df.index, res_df['zscore'], color=C['zscore'], lw=0.9, label='Z-score')
ax.axhline( Z_ENTRY, color=C['threshold'], lw=0.8, linestyle='--')
ax.axhline(-Z_ENTRY, color=C['threshold'], lw=0.8, linestyle='--', label=f'±{Z_ENTRY}')
ax.axhline(0,        color=C['zero'],      lw=0.5, linestyle=':')
ax.fill_between(res_df.index, res_df['zscore'], 0,
                where=(res_df['zscore'] < -Z_ENTRY), alpha=0.25, color=C['buy'])
ax.fill_between(res_df.index, res_df['zscore'], 0,
                where=(res_df['zscore'] >  Z_ENTRY), alpha=0.25, color=C['sell'])
ax.set_ylabel('Z-score', color='#aaaaaa', fontsize=9)
ax.legend(fontsize=7, loc='upper left', facecolor='#1a1a1a', edgecolor='#333', labelcolor='white')
ax.grid(alpha=0.08)

# ── Panel 3: RSI + ADX ───────────────────────────────────────
ax  = axes[2]
ax2 = ax.twinx()
ax.plot(res_df.index, res_df['rsi'], color=C['rsi'],  lw=0.9, label='RSI')
ax.axhline(RSI_OVERSOLD,   color=C['buy'],  lw=0.6, linestyle='--', alpha=0.6)
ax.axhline(RSI_OVERBOUGHT, color=C['sell'], lw=0.6, linestyle='--', alpha=0.6)
ax.axhline(50,             color=C['zero'], lw=0.5, linestyle=':')
ax2.plot(res_df.index, res_df['adx'], color=C['adx'], lw=0.9, linestyle='--', alpha=0.7, label='ADX')
ax2.axhline(ADX_TREND_THRESH, color=C['adx'], lw=0.6, linestyle=':', alpha=0.5)
ax.set_ylabel('RSI', color='#aaaaaa', fontsize=9)
ax2.set_ylabel('ADX', color='#aaaaaa', fontsize=9)
ax.set_ylim(0, 100)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=7, loc='upper left',
          facecolor='#1a1a1a', edgecolor='#333', labelcolor='white')
ax.grid(alpha=0.08)

# ── Panel 4: Equity vs Benchmark ─────────────────────────────
ax = axes[3]
ax.plot(res_df.index, res_df['equity'], color=C['equity'], lw=1.5, label='Strategy')
ax.plot(res_df.index, benchmark,        color=C['bench'],  lw=1.0, linestyle='--', alpha=0.7, label='Buy & Hold')
ax.axhline(1.0, color='#444', lw=0.6, linestyle=':')
ax.fill_between(res_df.index, res_df['equity'], benchmark,
                where=(res_df['equity'] >= benchmark), alpha=0.12, color=C['buy'],  label='Outperform')
ax.fill_between(res_df.index, res_df['equity'], benchmark,
                where=(res_df['equity'] <  benchmark), alpha=0.12, color=C['sell'], label='Underperform')
ax.set_ylabel('Equity (norm.)', color='#aaaaaa', fontsize=9)
ax.legend(fontsize=7, loc='upper left', facecolor='#1a1a1a', edgecolor='#333', labelcolor='white')
ax.grid(alpha=0.08)

# ── Panel 5: Drawdown ─────────────────────────────────────────
ax = axes[4]
ax.fill_between(res_df.index, res_df['drawdown'] * 100, 0,
                alpha=0.7, color=C['sell'], label='Drawdown')
ax.set_ylabel('Drawdown %', color='#aaaaaa', fontsize=9)
ax.set_ylim(top=0)
ax.legend(fontsize=7, loc='lower left', facecolor='#1a1a1a', edgecolor='#333', labelcolor='white')
ax.grid(alpha=0.08)

# ── Panel 6: Monte Carlo ──────────────────────────────────────
ax      = axes[5]
mc_days = np.arange(mc_equity.shape[0])

# Plot thin sample paths
sample_idx = np.random.choice(MC_SIMULATIONS, min(80, MC_SIMULATIONS), replace=False)
for idx in sample_idx:
    ax.plot(mc_days, mc_equity[:, idx], alpha=0.04, lw=0.5, color=C['mc'])

# Confidence bands
p5_path  = np.percentile(mc_equity, 5,  axis=1)
p50_path = np.percentile(mc_equity, 50, axis=1)
p95_path = np.percentile(mc_equity, 95, axis=1)

ax.fill_between(mc_days, p5_path, p95_path, alpha=0.2, color=C['mc'], label='P5–P95 band')
ax.plot(mc_days, p50_path, color='white',   lw=1.5, label='Median path')
ax.plot(mc_days, p5_path,  color=C['sell'], lw=1.0, linestyle='--', label='P5')
ax.plot(mc_days, p95_path, color=C['buy'],  lw=1.0, linestyle='--', label='P95')
ax.axhline(1.0, color='#444', lw=0.6, linestyle=':')
ax.set_ylabel('Equity (norm.)', color='#aaaaaa', fontsize=9)
ax.set_xlabel('Trading Days', color='#aaaaaa', fontsize=9)
ax.set_title(f'Monte Carlo ({MC_SIMULATIONS} bootstrapped paths)  |  '
             f'Median: {p50_path[-1]:.3f}x  |  '
             f'P5: {p5_path[-1]:.3f}x  |  '
             f'Prob profit: {prob_profit:.0%}',
             color='#aaaaaa', fontsize=9, pad=6)
ax.legend(fontsize=7, loc='upper left', facecolor='#1a1a1a', edgecolor='#333', labelcolor='white')
ax.grid(alpha=0.08)

plt.suptitle(f'Mean Reversion Backtest  —  {SYMBOL}',
             fontsize=15, fontweight='bold', color='white', y=1.005)
plt.savefig('mean_reversion_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Dashboard saved as mean_reversion_dashboard.png')